In [3]:
from pgpelib import PGPE
import numpy as np
from numbers import Real
from transformers import CLIPProcessor, CLIPModel
import torch
from sentence_transformers import SentenceTransformer
from PIL import Image
import random
import requests
from io import BytesIO
from sentence_transformers import SentenceTransformer
import time
from matplotlib import cm
import matplotlib.pyplot as plt
from tqdm import tqdm
from os import path
from sklearn.cluster import DBSCAN
from sentence_transformers import CrossEncoder

# Utils imports
from utils.rasterize import rasterize_shapes



DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

WIDTH = 300
HEIGHT = 300

CONTENT_DIR = './data'
PRIMITIVES_DIR = path.join(CONTENT_DIR, 'primitives')
RESULTS_DIR = path.join(CONTENT_DIR, 'results')

# Dict is better for multiple primitives pointing to same image
# e.g. 'person' -> 'person.png' & 'man' -> 'person.png'
PRIMITIVES_IMG_PATHS = {
    'umbrella': path.join(PRIMITIVES_DIR, 'umbrella.png'),
    'person':   path.join(PRIMITIVES_DIR, 'person.png'),
    'car':      path.join(PRIMITIVES_DIR, 'car.png'),
    'street':   path.join(PRIMITIVES_DIR, 'street.png'),
    'glass':    path.join(PRIMITIVES_DIR, 'glass.png'),
    'flame':    path.join(PRIMITIVES_DIR, 'flame.png'),
}

PRIMITIVES = np.array(list(PRIMITIVES_IMG_PATHS.keys()))


In [4]:
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(DEVICE)
def CLIP_emb_from_IMG(IMG):
  inputs = processor(images=IMG, return_tensors="pt", padding=True).to(DEVICE)

  with torch.no_grad():
    image_features = model.get_image_features(**inputs)
    image_features = image_features / image_features.norm(dim=1, keepdim=True)

  return image_features

def CLIP_emb_from_TEXT(TEXT):
  inputs = processor(text=TEXT, return_tensors="pt", padding=True).to(DEVICE)

  with torch.no_grad():
    text_features = model.get_text_features(**inputs)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)
  return text_features


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [5]:
PRIMITIVES_IMGS = {
    k: Image.open(v).resize((WIDTH, HEIGHT))
    for k, v in PRIMITIVES_IMG_PATHS.items()
}

PRIMITIVES_IMG_EMBEDDED = {
    k: CLIP_emb_from_IMG(np.array([PRIMITIVES_IMGS[k].convert('RGB')])).cpu().numpy()
    for k in PRIMITIVES_IMG_PATHS
}

In [39]:
# Reload modules automatically
%load_ext autoreload
%autoreload 2

In [15]:

def getMostSimilarPrimitivesWithRandomPI(sentence):
  alpha = 0.99/len(PRIMITIVES)
  print(alpha)
  model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
  sentenceEmb = model.encode(sentence)
  primitivesEmb = model.encode(PRIMITIVES)
  similarities = model.similarity(sentenceEmb, primitivesEmb)
  print(similarities)
  SM = torch.softmax(similarities[0], dim=0)
  indices = np.where(SM > alpha)[0]
  print([PRIMITIVES[i] for i in indices])
  return [(i,(random.uniform(0.01, 1),random.uniform(0.01, 1),random.uniform(0.01, 1),random.uniform(0.01, 1))) for i in indices] # X,Y,S,R


In [16]:
# Will return the best primitives for the given sentence using dynamic clustering
def shape_selector_clustering(text, eps=0.3, min_samples=2):
    # Compute the CLIP text embedding
    text_embedding = CLIP_emb_from_TEXT(text)  # shape: (1, embedding_dim)
    text_embedding_np = text_embedding.cpu().numpy().squeeze()
    
    # Compute cosine similarities (as dot product since embeddings are normalized)
    # Assuming PRIMITIVES_IMG_EMBEDDED is a dict with embeddings stored as tensors of shape (1, embedding_dim)
    shape_embeddings_np = np.array(list(PRIMITIVES_IMG_EMBEDDED.values())).squeeze(1)
    print("Shape embeddings shape:", shape_embeddings_np.shape)
    similarities = np.dot(shape_embeddings_np, text_embedding_np)
    
    # Use DBSCAN to cluster the shape embeddings dynamically
    dbscan = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
    cluster_labels = dbscan.fit_predict(shape_embeddings_np)
    
    selected_shape_ids = []
    
    # Get unique clusters (ignore noise labeled as -1)
    unique_clusters = [label for label in np.unique(cluster_labels) if label != -1]
    
    # For each cluster, find the shape with the highest similarity to the text embedding.
    for cluster in unique_clusters:
        # Indices of shapes in the current cluster
        cluster_indices = np.where(cluster_labels == cluster)[0]
        # Get similarities for shapes in the cluster
        cluster_similarities = similarities[cluster_indices]
        # Find index within cluster with max similarity
        best_idx_within_cluster = cluster_indices[np.argmax(cluster_similarities)]
        selected_shape_ids.append(best_idx_within_cluster)
    
    # Optionally, include noise points if needed (e.g., shapes that did not form any cluster)
    # You can sort these by similarity and pick a number of them if desired.
    noise_indices = np.where(cluster_labels == -1)[0]
    if len(noise_indices) > 0:
        noise_similarities = similarities[noise_indices]
        # For example, add the top 1 noise shape if its similarity is above a threshold
        best_noise_idx = noise_indices[np.argmax(noise_similarities)]
        # Adjust this condition as necessary:
        if similarities[best_noise_idx] > 0.3:
            selected_shape_ids.append(best_noise_idx)
    
    return selected_shape_ids

In [25]:
prompt_text = 'a person in the center of the image with an umbrella covering his head'
prompt_embedding = CLIP_emb_from_TEXT(prompt_text)
#PIs = getMostSimilarPrimitivesWithRandomPI(promptTEXT)
primitives_selected = ['person', 'umbrella']
primitives_selected_np = [PRIMITIVES_IMG_NP[i] for i in primitives_selected]
# Initial solution
x0 = np.random.uniform(0, 1, (len(primitives_selected), 4)) # 4 -> x, y, h, w

# TODO: normalize x0

pgpe = PGPE(

    # Length of each solution:
    solution_length=len(x0.flatten()),

    # Population size:
    popsize=256,

    # Initial center solution (i.e. initial mean of the search
    # distribution):
    center_init=x0.flatten(),

    # Learning rate for when updating the mean of the search distribution:
    center_learning_rate=0.001,

    # The following configuration tells that the ClipUp optimizer
    # is to be used when updating the mean of the search distribution.
    optimizer='clipup',

    # The ClipUp-specific 'max_speed' hyperparameter is set here:
    optimizer_config={'max_speed': 0.006},

    # If instead of ClipUp you would like to use Adam,
    # then set:
    #     optimizer='adam'
    # and do not specify any optimizer_config so that Adam is used
    # with its default hyperparameters.
    # For customizing how Adam works, you can also set
    # optimizer_config={'beta1': ..., 'beta2': ..., 'epsilon': ...}

    # Initial standard deviation of the search distribution
    stdev_init=0.1, # 1 ==> 0.01 per far si che gli update siamo più piccoli, ho provato 0.5 e cambia molto

    # Learning rate for when updating the standard deviation of the
    # search distribution:
    stdev_learning_rate=0.003,

    # With the setting below, the standard deviation cannot change
    # more tha then 20% of its original value.
    stdev_max_change=0.1, ###############

    # When estimating the gradient, rank the solutions linearly
    # (the ranks go from -0.5 to 0.5 from worst to best, like done
    # in the work of Salimans et al. (2017)):
    solution_ranking=True,

    # The PGPE solver will work with arrays of the following dtype:
    dtype='float32'
)

In [26]:

num_iterations = 5000
report_interval = 10
i = 0

for i in tqdm(range(1, 1 + num_iterations)):
    a = time.time()
    solutions = pgpe.ask()

    rasterized_imgs = []

    b = time.time()
    for it, solution in enumerate(solutions):
      # Reshape (one for each primitive)
      solution = solution.reshape(-1, 4)
      img = rasterize_shapes(primitives_selected_np, solution.reshape(-1, 4))
      rasterized_imgs.append(img)

    rasterized_imgs = np.stack(rasterized_imgs, axis=0) # (population_size, 300, 300, 4)
    rasterized_imgs = torch.tensor(rasterized_imgs).permute(0, 3, 1, 2).to(DEVICE)
    c = time.time()
    rasterized_emb = CLIP_emb_from_IMG(rasterized_imgs) # (population_size, 512)

    losses = torch.nn.functional.cosine_similarity(rasterized_emb, prompt_embedding, dim=1)

    t = time.time()
    print(f"ALL: {b-a}")
    print(f"SOL: {c-b}")
    print(f"EMB: {t-c}")
    print(f'best loss = {losses.max()}')
    print(f'avg loss = {np.average(losses)}')
    # if i % 10 == 0:
    #   print(f'loss migliore = {losses.max()}')
    #   best = solutions[losses.argmax()]
    #   print(f'best solution = {best}')
    #   img = IMGS[losses.argmax()].cpu().numpy()
    #   img = np.transpose(img, (1,2,0))
    img_pil = Image.fromarray(img.astype('uint8'), 'RGB')
    #   img_pil.save(path.join(RESULTS_DIR, f'img{i}.png'))
    display(img_pil)

    losses = losses.cpu()
    pgpe.tell(losses)


  0%|          | 0/5000 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [122]:
# Function that testes the rasterize function
def test_rasterize():
    for i in range(1000):
        # Generate random images
        num_images = np.random.randint(1, 5)
        image_list = []
        for _ in range(num_images):
            height = np.random.randint(50, 100) + 1
            width = height
            image = np.random.randint(0, 256, (height, width, 4), dtype=np.uint8)
            image_list.append(image)

        # Generate random positions
        num_positions = np.random.randint(1, 5)
        positions = []
        for _ in range(num_positions):
            x = np.random.rand()
            y = np.random.rand()
            h = np.random.rand()
            w = np.random.rand()
            positions.append((x, y, h, w))

        # Rasterize the images
        canvas = rasterize(image_list, positions)
        print(f"Canvas shape: {canvas.shape}")


In [26]:
test_rasterize()

Canvas shape: (92, 92, 4)
Canvas shape: (93, 93, 4)
Canvas shape: (80, 80, 4)
Canvas shape: (67, 67, 4)
Canvas shape: (61, 61, 4)
Canvas shape: (65, 65, 4)
Canvas shape: (76, 76, 4)
Canvas shape: (77, 77, 4)
Canvas shape: (54, 54, 4)
Canvas shape: (86, 86, 4)
Canvas shape: (89, 89, 4)
Canvas shape: (88, 88, 4)
Canvas shape: (75, 75, 4)
Canvas shape: (62, 62, 4)
Canvas shape: (87, 87, 4)
Canvas shape: (91, 91, 4)
Canvas shape: (61, 61, 4)
Canvas shape: (56, 56, 4)
Canvas shape: (96, 96, 4)
Canvas shape: (93, 93, 4)
Canvas shape: (94, 94, 4)
Canvas shape: (64, 64, 4)
Canvas shape: (51, 51, 4)
Canvas shape: (84, 84, 4)
Canvas shape: (67, 67, 4)
Canvas shape: (69, 69, 4)
Canvas shape: (51, 51, 4)
Canvas shape: (63, 63, 4)
Canvas shape: (94, 94, 4)
Canvas shape: (98, 98, 4)
Canvas shape: (69, 69, 4)
Canvas shape: (65, 65, 4)
Canvas shape: (71, 71, 4)
Canvas shape: (95, 95, 4)
Canvas shape: (91, 91, 4)
Canvas shape: (84, 84, 4)
Canvas shape: (73, 73, 4)
Canvas shape: (88, 88, 4)
Canvas shape

In [27]:
# Load image as numpy array
img1 = np.array(Image.open(PRIMITIVES_IMG_PATHS['person']))
img2 = np.array(Image.open(PRIMITIVES_IMG_PATHS['flame']))
img3 = np.array(Image.open(PRIMITIVES_IMG_PATHS['book']))


# X, Y, H, W
pos1 = (0.1, 0.1, 0.8, 0.8)
pos2 = (0.25, 0.4, 0.2, 0.2)
pos3 = (0.55, 0.4, 0.2, 0.2)

target = rasterize([img1, img2, img3], [pos1, pos2, pos3])
Image.fromarray(target)

NameError: name 'np' is not defined